Home Depot Search Relevance - NLP Final Project
------------------------------------------------
Pipeline:
  0. Setup & load
  1. Data hygiene (missing values, duplicates, relevance distribution)
  2. Text cleaning + unit/number normalization
  3. Search-type segmentation (code / dimension / brand / typo / generic)
  4. Qwen embeddings (cached to disk)
  5. Match scores (cosine): search vs title, search vs description
  6. Agreement quadrants (human relevance vs match score) + example rows
  7. Product clusters (categories) and their relevance
  8. Number & code matching (the model-number story)
  9. Exact-match (TF-IDF) vs embeddings, by search type
 10. Description / attribute "rescue" analysis
 11. Optional: does unit normalization raise the match score?
 12. Slide charts + summary tables

HOW TO RUN
  1. Put this file in the same folder as the Kaggle files (train.csv.zip, test.csv.zip,
     product_descriptions.csv.zip, attributes.csv.zip). No need to unzip them.
  2. From that folder:  uv run home_depot_search_analysis.py
     (uv reads the dependency block at the top and installs everything automatically.)
     Without uv: pip install the packages listed above, then python home_depot_search_analysis.py
     Or run the # %% cells one by one in VS Code/Jupyter.
  Results land in an "outputs" subfolder next to the script.

Runtime: everything except section 4 takes about a minute. Section 4 (Qwen embeddings,
~120k strings) is slow on a laptop CPU (expect a few hours); a GPU, e.g. free Colab T4,
does it in roughly 15-30 min. Embeddings are cached in outputs/, so this only happens once.
If you change the data or the model, delete outputs/emb_*.npy so they are rebuilt.

In [ ]:
# %% [0] Setup & load
import os
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

# Data folder = the folder this script sits in (works for `python script.py` and for
# VS Code / Jupyter cells, where __file__ is not defined and the cwd is used instead).
try:
    DATA_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    DATA_DIR = os.getcwd()
DATA_DIR = os.environ.get("HD_DATA_DIR", DATA_DIR)   # optional override
OUT_DIR = os.path.join(DATA_DIR, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)
ENC = "ISO-8859-1"           # the Kaggle files are not UTF-8
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)


def load(name):
    """Read name.csv, or name.csv.zip if it was never unzipped (pandas reads zips directly)."""
    for path in (f"{DATA_DIR}/{name}.csv", f"{DATA_DIR}/{name}.csv.zip"):
        if os.path.exists(path):
            return pd.read_csv(path, encoding=ENC)
    raise FileNotFoundError(f"Could not find {name}.csv or {name}.csv.zip in {DATA_DIR}")


train = load("train")
test = load("test")
desc = load("product_descriptions")
attrs = load("attributes")

# Only train has human relevance scores -> it drives every quality metric.
# Test is still useful for "what do customers search for" (search-type shares).

In [ ]:
# %% [1] Data hygiene
for name, df in [("train", train), ("test", test), ("descriptions", desc), ("attributes", attrs)]:
    print(f"\n{name}: {df.shape}")
    print(df.isna().sum().to_string())

attrs = attrs.dropna(subset=["product_uid"]).copy()
attrs["product_uid"] = attrs["product_uid"].astype(int)

print("\nDuplicate ids:", train["id"].duplicated().sum())
print("Duplicate (search, product) pairs:", train.duplicated(["search_term", "product_uid"]).sum())
print("Train products missing a description:", (~train["product_uid"].isin(desc["product_uid"])).sum())
print("Share of train rows whose product has attributes:",
      round(train["product_uid"].isin(attrs["product_uid"]).mean(), 3))
print("Test search terms also seen in train:",
      round(test["search_term"].isin(train["search_term"]).mean(), 3))

print("\nRelevance summary (every row is a result search actually returned):")
print(train["relevance"].describe().round(3).to_string())
print(train["relevance"].round(2).value_counts().sort_index().to_string())

In [ ]:
# %% [2] Cleaning + normalization
def clean_html(s: str) -> str:
    s = re.sub(r"<[^>]+>", " ", str(s))
    s = re.sub(r"([a-z])([A-Z])", r"\1 \2", s)  # descriptions often glue sentences: "doorWe"
    return s


def _frac_to_dec(m):
    whole = int(m.group(1)) if m.group(1) else 0
    num, den = int(m.group(2)), int(m.group(3))
    return f"{whole + num / den:g}" if den else m.group(0)


def normalize(s: str) -> str:
    """Lowercase + make numbers/units comparable: '1-1/2 in.' -> '1.5 in', '2x4' -> '2 x 4'."""
    s = str(s).lower()
    s = re.sub(r"(?:(\d+)[-\s])?(\d+)/(\d+)", _frac_to_dec, s)               # fractions -> decimals
    s = re.sub(r'(\d)\s*-?\s*(?:"|inches|inch|in\.|in\b)', r"\1 in ", s)
    s = re.sub(r"(\d)\s*-?\s*(?:'|feet|foot|ft\.?)", r"\1 ft ", s)
    s = re.sub(r"(\d)\s*-?\s*(?:pounds|pound|lbs\.?|lb\.?)", r"\1 lb ", s)
    s = re.sub(r"(\d)\s*-?\s*(?:gallons|gallon|gal\.?)", r"\1 gal ", s)
    s = re.sub(r"(\d)\s*-?\s*(?:volts|volt|v)\b", r"\1 volt ", s)
    s = re.sub(r"(\d)\s*-?\s*(?:watts|watt|w)\b", r"\1 watt ", s)
    s = re.sub(r"(\d)\s*-?\s*(?:amps|amp)\b", r"\1 amp ", s)
    s = re.sub(r"(\d)\s*-?\s*(mm|cm)\b", r"\1 \2 ", s)
    s = re.sub(r"(\d)\s*[x×*]\s*(?=\d)", r"\1 x ", s)                        # 2x4x8 -> 2 x 4 x 8
    s = re.sub(r"[^a-z0-9.\- ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


desc["desc_clean"] = desc["product_description"].map(clean_html)
train = train.merge(desc[["product_uid", "desc_clean"]], on="product_uid", how="left")
train["desc_clean"] = train["desc_clean"].fillna("")

train["q_norm"] = train["search_term"].map(normalize)
train["t_norm"] = train["product_title"].map(normalize)
train["d_norm"] = train["desc_clean"].map(normalize)
test["q_norm"] = test["search_term"].map(normalize)

In [ ]:
# %% [3] Search-type segmentation
DIM_RE = re.compile(r"\d+(?:\.\d+)?\s(?:in|ft|lb|gal|volt|watt|amp|mm|cm)\b|\d x \d")

# Vocabulary of "real" words from the whole catalog (all titles + all descriptions)
word_counts = Counter()
for text in pd.concat([train["product_title"], test["product_title"], desc["desc_clean"]]).str.lower():
    word_counts.update(re.findall(r"[a-z]+", text))
vocab = {w for w, c in word_counts.items() if c >= 3}

TOKEN_RE = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*")

# A number followed by a unit abbreviation is a measurement, not a model number: 12oz, 30a, 18cu, 16ga
UNIT_TOKEN_RE = re.compile(
    r"\d+(?:oz|a|ga|ah|cu|hp|psi|cfm|btu|gpm|gpf|awg|mil|qt|pk|pc|pcs|ct|lm|yd|yds|sq|mah|kw|"
    r"v|w|g|l|m|k|x|lb|lbs|ft|in|mm|cm|gal|amp|amps|volt|watt|pack|piece)")
# Short industry spec designations: bulb shapes/bases (par38, mr16, br30, e26, gu24, t12),
# insulation R-values (r13), battery platforms (m18), respirators (n95). These describe a TYPE of
# product, not one specific item, so they are counted as size/spec searches, not model numbers.
SPEC_TOKEN_RE = re.compile(r"[a-z]{1,3}\d{1,3}[a-z]?")


def is_code(token: str) -> bool:
    """Model/part number that identifies ONE product: 'wrt111sfdb', 'lw5200e', 'hb4190', '204220439'.
    Not: '3-way', '18volt' (digits + ordinary word), '12000' (round spec numbers like BTU),
    '5000k' (color temperature), '12oz'/'30a' (units), 'par38'/'m18' (spec designations),
    or anything shorter than 5 characters."""
    compact_tok = token.replace("-", "")
    if len(compact_tok) < 5 or UNIT_TOKEN_RE.fullmatch(compact_tok) or SPEC_TOKEN_RE.fullmatch(compact_tok):
        return False
    digits = sum(ch.isdigit() for ch in token)
    if token.isdigit():
        return not token.endswith("000")
    if digits < 2 or not any(ch.isalpha() for ch in token):
        return False
    letter_runs = re.findall(r"[a-z]+", token)
    return all(len(r) <= 3 or r not in vocab for r in letter_runs)


def _tokens(q: str) -> list:
    # Remove decimals first: normalize() turns "1/3" into "0.333333", whose digits must not
    # be mistaken for a part number.
    return TOKEN_RE.findall(re.sub(r"\d*\.\d+", " ", q))


def codes_in(q: str) -> list:
    return [t for t in _tokens(q) if is_code(t)]


def specs_in(q: str) -> list:
    return [t for t in _tokens(q) if SPEC_TOKEN_RE.fullmatch(t) and t not in vocab]


# Typo detection: a word is a likely misspelling if it is NOT in the catalog vocabulary but
# IS within ~2 edits of a catalog word ("curtiins" -> "curtains"), or is two catalog words
# run together ("carpetshampoo"). Unknown words with no close match are left alone.
def _deletes(w):
    return {w[:i] + w[i + 1:] for i in range(len(w))}


common_vocab = {w for w, c in word_counts.items() if c >= 20 and len(w) >= 3}
delete_index = set()
for w in common_vocab:
    delete_index |= _deletes(w)


def is_typo(w: str) -> bool:
    if w in vocab or len(w) < 4:
        return False
    if any(w[:i] in common_vocab and w[i:] in common_vocab for i in range(3, len(w) - 2)):
        return True                                       # missing space
    d = _deletes(w)
    return w in delete_index or bool(d & common_vocab) or bool(d & delete_index)


def ngrams(tokens, n_max=3):
    return {" ".join(tokens[i:i + n]) for n in range(1, n_max + 1) for i in range(len(tokens) - n + 1)}

In [ ]:
# --- Brand list --------------------------------------------------------------------
# Source: the "MFG Brand Name" attribute (the manufacturer brand Home Depot stores per product).
brand_of = (attrs[attrs["name"] == "MFG Brand Name"].dropna(subset=["value"])
            .drop_duplicates("product_uid")
            .set_index("product_uid")["value"].str.lower().str.strip())
brands = {b for b in brand_of.unique()
          if len(b) >= 4 and b not in {"unbranded", "generic", "none", "n/a"}}

# Remove "brands" that are really ordinary words. Some small brands are named "Impact", "Square",
# "Dual", "Edge", "Finish", "Universal"... Matching those would tag "impact driver" or
# "dual wall duct" as a brand search.
# Test: inside product descriptions, real brand names are written Capitalized ("DeWalt", "Glidden"),
# while ordinary words appear in lowercase mid-sentence ("square feet", "dual"). A single-word
# brand that is lowercase more than half the time is treated as an ordinary word.
case_counts = Counter()
for text in desc["product_description"].astype(str).str.replace(r"<[^>]+>", " ", regex=True):
    for sentence in re.split(r"[.!?:;]\s+", text):
        case_counts.update(re.findall(r"(?<=\s)[A-Za-z][A-Za-z'-]+", sentence))  # skips 1st word


def lowercase_share(word):
    lower = case_counts[word]
    capital = case_counts[word.capitalize()] + case_counts[word.upper()]
    return lower / (lower + capital) if lower + capital >= 20 else 0.0


generic_brands = {b for b in brands if " " not in b and lowercase_share(b) > 0.5}
brands -= generic_brands
print(f"Brands: {brand_of.nunique():,} in catalog | {len(brands):,} used | "
      f"{len(generic_brands)} dropped as ordinary words, e.g. {sorted(generic_brands, key=lambda b: -case_counts[b])[:12]}")


def segment(q: str) -> dict:
    tokens = q.split()
    typos = [w for w in re.findall(r"[a-z]+", q) if is_typo(w)]
    flags = {
        "is_code": bool(codes_in(q)),
        "is_dimension": bool(DIM_RE.search(q)) or bool(specs_in(q)),
        "is_brand": bool(ngrams(tokens) & brands),
        "is_typo": len(typos) > 0,
    }
    # One primary label per search for clean slides (priority order matters)
    for label, key in [("Model/product code", "is_code"), ("Misspelled", "is_typo"),
                       ("Size/spec", "is_dimension"), ("Brand", "is_brand")]:
        if flags[key]:
            flags["search_type"] = label
            break
    else:
        flags["search_type"] = "Generic"
    return flags


train = pd.concat([train, pd.DataFrame(train["q_norm"].map(segment).tolist(), index=train.index)], axis=1)
test_types = pd.DataFrame(test["q_norm"].map(segment).tolist())

seg_table = (train.groupby("search_type")
             .agg(rows=("relevance", "size"), mean_relevance=("relevance", "mean"))
             .assign(share_of_train=lambda d: d["rows"] / d["rows"].sum())
             .sort_values("mean_relevance"))
seg_table["share_of_test"] = test_types["search_type"].value_counts(normalize=True)
print("\nRelevance by search type:\n", seg_table.round(3))

# Spot-check the rules before trusting them
for t in seg_table.index:
    print(f"\n{t}:", train.loc[train["search_type"] == t, "search_term"].sample(5, random_state=1).tolist())

In [ ]:
# %% [4] Qwen embeddings (cached)
# The embeddings are saved in outputs/emb_*.npy. When those files exist, this section just loads
# them: the Qwen model is NOT downloaded and torch is NOT imported, so it also runs on a kernel
# without torch/sentence-transformers. The model only loads if a cache file is missing or stale.
MODEL_NAME = "Qwen/Qwen3-Embedding-8B"
_model = None
DEVICE = None


def get_model():
    """Load Qwen once, only when something actually needs embedding."""
    global _model, DEVICE
    if _model is None:
        try:
            import torch
            from sentence_transformers import SentenceTransformer
        except ImportError as e:
            raise ImportError(
                "An embedding cache file is missing, so Qwen is needed, but torch/sentence-transformers "
                "are not installed in this kernel. Either put the outputs/emb_*.npy files back, or install "
                "them (see the GPU setup notes)." ) from e
        if torch.cuda.is_available():
            DEVICE, dtype = "cuda", torch.bfloat16      # bf16: half the memory, faster on RTX 30/40/50
            print(f"GPU: {torch.cuda.get_device_name(0)} | torch {torch.__version__} | CUDA {torch.version.cuda}")
        elif torch.backends.mps.is_available():
            DEVICE, dtype = "mps", torch.float16        # Apple silicon
            print("GPU: Apple MPS")
        else:
            DEVICE, dtype = "cpu", torch.float32
            print(f"WARNING: no GPU found - running on CPU (slow). torch {torch.__version__}, "
                  f"CUDA build: {torch.version.cuda}. 'None' means a CPU-only torch was installed.")
        _model = SentenceTransformer(MODEL_NAME, device=DEVICE, model_kwargs={"torch_dtype": dtype})
        _model.max_seq_length = 256   # descriptions are long; first ~256 tokens carry the key info
        print("Embedding on:", _model.device)
    return _model


def embed(texts, name, query=False, long_texts=False):
    path = f"{OUT_DIR}/emb_{name}.npy"
    if os.path.exists(path):
        E = np.load(path)
        if len(E) == len(texts):
            print(f"Loaded cached embeddings: {path}")
            return E
        # The list of texts changed since this file was saved -> rows would be misaligned.
        print(f"Cache {path} has {len(E):,} rows but {len(texts):,} texts now - re-embedding.")
    model = get_model()
    # Batch sizes tuned for a 32 GB card (RTX 5090). Halve on "CUDA out of memory".
    batch_size = (128 if long_texts else 256) if DEVICE == "cuda" else (16 if long_texts else 32)
    kwargs = dict(batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True)
    if query:
        kwargs["prompt_name"] = "query"   # Qwen3-Embedding: instruction prefix on queries only
    E = model.encode(list(texts), **kwargs).astype(np.float32)
    np.save(path, E)
    return E


# Embed unique strings once, then map back to rows. Embed the original wording
# (lightly cleaned) so cosine reflects what the search engine actually sees.
uq = pd.Index(train["search_term"].str.lower().unique())
ut = pd.Index(train["product_title"].unique())
products = train.drop_duplicates("product_uid")[["product_uid", "product_title", "desc_clean"]].reset_index(drop=True)
up = pd.Index(products["product_uid"])

Q = embed(uq, "queries", query=True)
T = embed(ut, "titles")
D = embed(products["desc_clean"].str.slice(0, 1500), "descriptions", long_texts=True)

qi = uq.get_indexer(train["search_term"].str.lower())
ti = ut.get_indexer(train["product_title"])
di = up.get_indexer(train["product_uid"])

In [ ]:
# %% [5] Match scores (cosine) per row
def row_cos(A, ia, B, ib):
    return np.einsum("ij,ij->i", A[ia], B[ib])


train["cos_title"] = row_cos(Q, qi, T, ti)
train["cos_desc"] = row_cos(Q, qi, D, di)
train["cos_best"] = train[["cos_title", "cos_desc"]].max(axis=1)

rho, _ = spearmanr(train["cos_title"], train["relevance"])
print(f"\nSpearman(relevance, title match score) = {rho:.3f}")

In [ ]:
# %% [6] Agreement quadrants: human relevance vs match score
hi_cos, lo_cos = train["cos_title"].quantile([0.75, 0.25])
rel_band = np.select([train["relevance"] >= 2.5, train["relevance"] <= 2.0], ["High relevance", "Low relevance"], "Mid")
cos_band = np.select([train["cos_title"] >= hi_cos, train["cos_title"] <= lo_cos], ["High match", "Low match"], "Mid")
train["quadrant"] = pd.Series(rel_band, index=train.index) + " / " + pd.Series(cos_band, index=train.index)

quad = train[~train["quadrant"].str.contains("Mid")]
print("\nQuadrant sizes:\n", quad["quadrant"].value_counts())
print("\nSearch-type mix per quadrant:\n",
      pd.crosstab(quad["quadrant"], quad["search_type"], normalize="index").round(2))

# Concrete examples for slides (executives remember examples, not statistics)
examples = (quad.sample(frac=1, random_state=0).groupby("quadrant").head(15)
            .sort_values("quadrant")
            [["quadrant", "search_term", "product_title", "relevance", "cos_title", "search_type"]])
examples.to_csv(f"{OUT_DIR}/quadrant_examples.csv", index=False)

In [ ]:
# %% [7] Product clusters ("categories") and their relevance
K = 40   # 20 was too coarse (one catch-all cluster); 40 separates categories better
km = KMeans(n_clusters=K, n_init=4, random_state=42)
prod_title_idx = ut.get_indexer(products["product_title"])
products["cluster"] = km.fit_predict(T[prod_title_idx])

# Name clusters by their most distinctive title words
cluster_docs = products.groupby("cluster")["product_title"].apply(" ".join)
tf = TfidfVectorizer(stop_words="english", max_features=20000, token_pattern=r"[a-zA-Z]{3,}")
Xc = tf.fit_transform(cluster_docs)
terms = np.array(tf.get_feature_names_out())
cluster_names = {c: ", ".join(terms[Xc[i].toarray().ravel().argsort()[::-1][:4]])
                 for i, c in enumerate(cluster_docs.index)}

train["cluster"] = train["product_uid"].map(products.set_index("product_uid")["cluster"])
train["cluster_name"] = train["cluster"].map(cluster_names)

cluster_table = (train.groupby("cluster_name")
                 .agg(rows=("relevance", "size"),
                      mean_relevance=("relevance", "mean"),
                      mean_match=("cos_title", "mean"),
                      code_search_share=("is_code", "mean"),
                      typo_share=("is_typo", "mean"))
                 .query("rows >= 300")              # avoid tiny-cluster noise
                 .sort_values("mean_relevance"))
print("\nCategories by relevance (worst first):\n", cluster_table.round(3))

In [ ]:
# %% [8] Number & code matching (model-number story)
NUM_RE = re.compile(r"\d+(?:\.\d+)?")


def number_match(q, t):
    qn = set(NUM_RE.findall(q))
    if not qn:
        return "No numbers in search"
    hit = len(qn & set(NUM_RE.findall(t)))
    return "All numbers match" if hit == len(qn) else ("Some match" if hit else "No numbers match")


train["number_match"] = [number_match(q, t) for q, t in zip(train["q_norm"], train["t_norm"])]
print("\nRelevance by number match:\n",
      train.groupby("number_match")["relevance"].agg(["size", "mean"]).round(3))

# Where does the searched code actually live? title / description / attributes / nowhere
attr_text = (attrs[attrs["product_uid"].isin(train["product_uid"])]
             .assign(v=lambda d: d["value"].fillna("").astype(str).str.lower())
             .groupby("product_uid")["v"].apply(" ".join))
compact = lambda s: re.sub(r"[\s\-]", "", str(s).lower())
train["title_c"] = train["t_norm"].map(compact)
train["desc_c"] = train["d_norm"].map(compact)
train["attr_c"] = train["product_uid"].map(attr_text).fillna("").map(compact)


def code_location(row):
    codes = [c.replace("-", "") for c in codes_in(row.q_norm)]
    if not codes:
        return None
    for col, label in [("title_c", "In title"), ("desc_c", "In description"), ("attr_c", "In attributes only")]:
        if any(c in getattr(row, col) for c in codes):
            return label
    return "Not found on product"


train["code_location"] = train.apply(code_location, axis=1)
code_rows = train[train["is_code"]]
print("\nCode searches - where the code is found:\n",
      code_rows.groupby("code_location")
      .agg(rows=("relevance", "size"), mean_relevance=("relevance", "mean"), mean_match=("cos_title", "mean"))
      .round(3))

# Model-number attributes: do they exist, and are they searchable?
model_attr_names = attrs.loc[attrs["name"].str.contains("model", case=False, na=False), "name"].value_counts()
print("\nAttribute names mentioning 'model':\n", model_attr_names.head(10))

# Catalog coverage: does each searched model number exist ANYWHERE in the catalog
# (any product title in train/test, or any attribute value)? If not, search cannot find it.
catalog_titles = " ".join(pd.concat([train["product_title"], test["product_title"]]).drop_duplicates().map(
    lambda s: compact(normalize(s))))
catalog_attrs = " ".join(attrs["value"].dropna().astype(str).map(lambda s: compact(normalize(s))))
searched_models = sorted({c.replace("-", "") for q in code_rows["q_norm"] for c in codes_in(q)})
in_catalog = [m for m in searched_models if m in catalog_titles or m in catalog_attrs]
model_coverage_pct = 100 * len(in_catalog) / max(len(searched_models), 1)
print(f"\nDistinct model numbers customers searched: {len(searched_models)} | "
      f"found anywhere in the catalog (titles or specs): {len(in_catalog)} ({model_coverage_pct:.0f}%)")

In [ ]:
# %% [9] Exact matching (TF-IDF) vs embeddings, by search type
lex = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2)
lex.fit(pd.concat([pd.Series(train["q_norm"].unique()), pd.Series(train["t_norm"].unique())]))
Ql, Tl = lex.transform(train["q_norm"]), lex.transform(train["t_norm"])
train["cos_exact"] = np.asarray(Ql.multiply(Tl).sum(axis=1)).ravel()

rows = []
for t, g in train.groupby("search_type"):
    rows.append({
        "search_type": t,
        "rows": len(g),
        "embedding_vs_relevance": spearmanr(g["cos_title"], g["relevance"])[0],
        "exact_match_vs_relevance": spearmanr(g["cos_exact"], g["relevance"])[0],
    })
method_table = pd.DataFrame(rows).set_index("search_type").round(3)
print("\nWhich match score tracks human judgment better? (Spearman)\n", method_table)
# Expectation to test: exact matching wins for code searches -> supports a hybrid / exact-match lane.

In [ ]:
# %% [10] Description rescue: when the title match is weak, does the description help?
weak = train[train["cos_title"] <= lo_cos]
print(f"\nWeak title matches: {len(weak)} rows, mean relevance {weak['relevance'].mean():.3f}")
print("Share where the description matches better than the title:",
      round((weak["cos_desc"] > weak["cos_title"]).mean(), 3))
print("Mean relevance when description beats title vs not:\n",
      weak.groupby(weak["cos_desc"] > weak["cos_title"])["relevance"].mean().round(3))
print("Spearman with relevance - title only:", round(spearmanr(train["cos_title"], train["relevance"])[0], 3),
      "| best of title/description:", round(spearmanr(train["cos_best"], train["relevance"])[0], 3))

In [ ]:
# %% [11] Optional: does normalizing units/numbers raise the match score?
RUN_NORMALIZATION_TEST = True   # uses cached outputs/emb_*_norm_dims.npy when present
if RUN_NORMALIZATION_TEST:
    # Measurement searches only ("27 in", "2 x 4", "60 amp"). Spec codes like PAR38 are left out:
    # unit normalization doesn't touch them, and this keeps the same search list the cached
    # emb_*_norm_dims.npy files were built from.
    dims = train[train["q_norm"].str.contains(DIM_RE)].copy()
    uqn, utn = pd.Index(dims["q_norm"].unique()), pd.Index(dims["t_norm"].unique())
    try:
        Qn = embed(uqn, "queries_norm_dims", query=True)
        Tn = embed(utn, "titles_norm_dims")
    except ImportError as e:
        print(f"\nSection 11 skipped: {e}")
        Qn = Tn = None
    if Qn is not None:
        dims["cos_title_norm"] = row_cos(Qn, uqn.get_indexer(dims["q_norm"]), Tn, utn.get_indexer(dims["t_norm"]))
        print("\nSize/dimension searches - mean match score raw vs normalized:",
              round(dims["cos_title"].mean(), 3), "->", round(dims["cos_title_norm"].mean(), 3))
        print("Spearman with relevance raw vs normalized:",
              round(spearmanr(dims["cos_title"], dims["relevance"])[0], 3), "->",
              round(spearmanr(dims["cos_title_norm"], dims["relevance"])[0], 3))

In [ ]:
# %% [12] Slide charts + summary tables (plain language labels)
def barh(series, title, xlabel, fname, counts=None, label_fmt="  {v:.2f}  (n={n:,})"):
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(series) + 1.5))
    ax.barh(series.index.astype(str), series.values, color="#f96302")   # Home Depot orange
    ax.set_title(title, loc="left", fontsize=13, weight="bold")
    ax.set_xlabel(xlabel)
    if counts is not None:
        for i, (v, n) in enumerate(zip(series.values, counts.values)):
            ax.text(v, i, label_fmt.format(v=v, n=n), va="center", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/{fname}", dpi=200)
    plt.close(fig)


# Chart 01: every average rating that occurs, as its own labeled bar (categorical, so no empty
# rows), colored by band. Rare in-between averages (1.25, 1.75, ... from 4+ raters) are folded
# into the nearest common value so the chart stays readable.
COMMON = np.array([1.0, 1.33, 1.67, 2.0, 2.33, 2.67, 3.0])
rel_bucket = COMMON[np.abs(train["relevance"].to_numpy()[:, None] - COMMON[None, :]).argmin(axis=1)]
rel_dist = pd.Series(rel_bucket).value_counts().sort_index()
band_color = ["#3d3d3d" if v < 2 else ("#f96302" if v == 3 else "#d4d4d4") for v in rel_dist.index]
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh([f"{v:.2f}" for v in rel_dist.index], rel_dist.values, color=band_color, height=0.7)
for i, n in enumerate(rel_dist.values):
    ax.text(n, i, f"  {n:,} ({100 * n / rel_dist.sum():.0f}%)", va="center", fontsize=9)
poor_share, perfect_share = 100 * (train["relevance"] < 2).mean(), 100 * (train["relevance"] == 3).mean()
ax.set_title(f"Today: {poor_share:.0f}% of results are poor, {perfect_share:.0f}% are perfect",
             loc="left", fontsize=13, weight="bold")
ax.set_ylabel("Average rating (1 = not, 2 = mildly, 3 = highly relevant)")
ax.set_xlabel("Number of results")
ax.set_xlim(0, rel_dist.max() * 1.25)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, 0.005, "Dark = poor (average below 2). Orange = perfect (average of 3). Gray = okay.",
         fontsize=8, color="#6b6b6b")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/01_relevance_distribution.png", dpi=200, bbox_inches="tight")
plt.close(fig)

barh(seg_table["mean_relevance"], "Result quality by type of search", "Average relevance (1-3)",
     "02_relevance_by_search_type.png", seg_table["rows"])

nm = train.groupby("number_match")["relevance"].agg(["mean", "size"]).sort_values("mean")
barh(nm["mean"], "When the numbers match, results are better", "Average relevance (1-3)",
     "03_number_match.png", nm["size"])

ct = pd.concat([cluster_table.head(5), cluster_table.tail(5)])
barh(ct["mean_relevance"], "Weakest and strongest product categories", "Average relevance (1-3)",
     "04_categories.png", ct["rows"])

cl = code_rows["code_location"].value_counts().sort_values()
cl_pct = 100 * cl / cl.sum()
barh(cl_pct, "Model numbers are almost never on the product",
     f"Share of results (%)  -  only {model_coverage_pct:.0f}% of the {len(searched_models)} model numbers "
     f"searched exist anywhere in the catalog", "05_code_location.png", cl,
     label_fmt="  {v:.0f}%  ({n:,} results)")

with pd.ExcelWriter(f"{OUT_DIR}/summary_tables.xlsx") as xw:
    seg_table.to_excel(xw, sheet_name="search_types")
    cluster_table.to_excel(xw, sheet_name="categories")
    method_table.to_excel(xw, sheet_name="exact_vs_embedding")
    nm.to_excel(xw, sheet_name="number_match")
    examples.to_excel(xw, sheet_name="examples", index=False)

In [ ]:
# %% [13] Deep dive: exact vs missing product numbers -> how relevance is affected
# Averages hide the story (2.60 vs 2.14 sounds small on a 1-3 scale). Customers experience
# results as "useless", "okay" or "exactly right", so we report those SHARES instead:
# Relevance is the AVERAGE of at least three human ratings on a 1-3 scale
# (1 = not relevant, 2 = mildly relevant, 3 = highly relevant).
#   Poor    = average below 2.0 (falls short of "mildly relevant", leaning "not relevant")
#   Perfect = average of 3.0    (every rater scored it "highly relevant")
from difflib import SequenceMatcher
from scipy.stats import mannwhitneyu

train["poor"] = train["relevance"] < 2.0
train["perfect"] = train["relevance"] == 3.0


def outcome_table(df, by):
    return (df.groupby(by)
            .agg(rows=("relevance", "size"), mean_relevance=("relevance", "mean"),
                 poor_pct=("poor", "mean"), perfect_pct=("perfect", "mean"))
            .assign(poor_pct=lambda d: (100 * d["poor_pct"]).round(1),
                    perfect_pct=lambda d: (100 * d["perfect_pct"]).round(1))
            .round(3))


base = outcome_table(train.assign(all="All results"), "all")
print("\nBaseline (all results):\n", base)

In [ ]:
# 13a. Four levels of model-number match (adds a "near miss" level: same product family,
#      e.g. searched 'gt200r', product carries 'gt225')
code = train[train["is_code"]].copy()
code["codes"] = code["q_norm"].map(lambda q: [c.replace("-", "") for c in codes_in(q)])
NUMTOK_RE = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*")


def code_match_level(row):
    if any(c in row.title_c for c in row.codes):
        return "1. Exact number in title"
    if any(c in row.desc_c or c in row.attr_c for c in row.codes):
        return "2. Exact number in description/specs"
    text = f"{row.t_norm} {row.d_norm} {row.attr_c}"
    product_nums = {t.replace("-", "") for t in NUMTOK_RE.findall(re.sub(r"[./]", "", text))
                    if len(t) >= 4 and any(ch.isdigit() for ch in t)}
    for c in row.codes:
        for t in product_nums:
            if t[:4] == c[:4] or SequenceMatcher(None, c, t).ratio() >= 0.75:
                return "3. Near miss (similar number)"
    return "4. Number missing from product"


code["code_match"] = code.apply(code_match_level, axis=1)
code_match_table = outcome_table(code, "code_match")
print("\nModel-number searches by match level:\n", code_match_table)

exact = code.loc[code["code_match"].str.startswith("1"), "relevance"]
missing = code.loc[code["code_match"].str.startswith("4"), "relevance"]
print(f"Statistical test (exact vs missing): Mann-Whitney p = {mannwhitneyu(exact, missing).pvalue:.1e}")

In [ ]:
# 13b. Per SEARCH, not per result: did ANY returned product carry the number?
per_search = code.groupby("search_term").agg(
    results=("relevance", "size"),
    any_exact=("code_match", lambda s: s.str[0].isin(["1", "2"]).any()),
    best_relevance=("relevance", "max"),
    mean_relevance=("relevance", "mean"))
no_hit_share = 100 * (~per_search["any_exact"]).mean()
print(f"\nModel-number searches: {len(per_search)} | avg results each: {per_search['results'].mean():.1f}")
print(f"Share where NOT ONE returned product carries the number: {no_hit_share:.0f}%")
print(per_search.groupby("any_exact")[["best_relevance", "mean_relevance"]].mean().round(3))

In [ ]:
# 13c. Brand as a safety net: when the number is missing, did search at least get the brand right?
def searched_brand(q):
    toks = q.split()
    hits = ngrams(toks) & brands
    return max(hits, key=len) if hits else None


code["q_brand"] = code["q_norm"].map(searched_brand)
code["p_brand"] = code["product_uid"].map(brand_of).fillna("")


def brand_status(row):
    if not isinstance(row.q_brand, str):
        return "No brand in search"
    if row.q_brand in row.p_brand or (row.p_brand and row.p_brand in row.q_brand) or row.q_brand in row.t_norm:
        return "Right brand"
    return "Wrong brand"


code["brand_status"] = code.apply(brand_status, axis=1)
brand_table = outcome_table(code[code["code_match"].str.startswith("4")], "brand_status")
print("\nNumber missing - did search at least get the brand right?\n", brand_table)

In [ ]:
# 13d. Size/dimension searches (bigger volume than model numbers): right size vs wrong size
size = train[train["is_dimension"] & ~train["is_code"]].copy()
size["size_match"] = size["number_match"].map({"All numbers match": "Right size",
                                               "Some match": "Partly right size",
                                               "No numbers match": "Wrong size"})
size_table = outcome_table(size.dropna(subset=["size_match"]), "size_match")
print("\nSize/dimension searches:\n", size_table)

In [ ]:
# 13e. Is this a category problem or a search-wide problem? Mismatch penalty per category.
num = train[train["number_match"] != "No numbers in search"].copy()
num["numbers"] = np.where(num["number_match"] == "All numbers match", "match", "mismatch")
pen = num.groupby(["cluster_name", "numbers"])["relevance"].agg(["size", "mean"]).unstack()
pen = pen[(pen[("size", "match")] >= 30) & (pen[("size", "mismatch")] >= 30)]
penalty = pd.DataFrame({
    "match": pen[("mean", "match")], "mismatch": pen[("mean", "mismatch")],
    "n_match": pen[("size", "match")], "n_mismatch": pen[("size", "mismatch")]})
penalty["penalty"] = penalty["match"] - penalty["mismatch"]
penalty = penalty.sort_values("penalty", ascending=False).round(3)
print(f"\nCategories tested: {len(penalty)} | categories where mismatched numbers score worse: "
      f"{(penalty['penalty'] > 0).sum()} | median penalty: {penalty['penalty'].median():.2f}")
print(penalty.head(10))

In [ ]:
# 13f. Worst segments: category x number status (n >= 100)
seg = (train.assign(numbers=train["number_match"].replace({"Some match": "No numbers match"}))
       .groupby(["cluster_name", "numbers"])
       .agg(rows=("relevance", "size"), mean_relevance=("relevance", "mean"),
            poor_pct=("poor", "mean"), perfect_pct=("perfect", "mean")))
seg = seg[seg["rows"] >= 100].sort_values("mean_relevance")
seg[["poor_pct", "perfect_pct"]] = (100 * seg[["poor_pct", "perfect_pct"]]).round(1)
print("\nWorst 10 segments:\n", seg.head(10).round(3))

In [ ]:
# --- Charts for 13 ---------------------------------------------------------------
INK, MUTED = "#222222", "#6b6b6b"
OUTCOME_COLORS = {"Poor": "#3d3d3d", "Okay": "#d4d4d4", "Perfect": "#f96302"}


def outcome_bars(table, title, fname, note=None):
    """100% stacked bars: share of results that are Poor / Okay / Perfect."""
    t = table.copy()
    t["Okay"] = 100 - t["poor_pct"] - t["perfect_pct"]
    parts = {"Poor": t["poor_pct"], "Okay": t["Okay"], "Perfect": t["perfect_pct"]}
    labels = [f"{i}\n(n={n:,})" for i, n in zip(t.index, t["rows"])]
    fig, ax = plt.subplots(figsize=(9, 0.75 * len(t) + 1.9))
    left = np.zeros(len(t))
    for name, vals in parts.items():
        ax.barh(labels, vals.values, left=left, color=OUTCOME_COLORS[name], label=name,
                edgecolor="white", linewidth=2, height=0.62)
        for i, (l, v) in enumerate(zip(left, vals.values)):
            if name != "Okay" and v >= 4:
                ax.text(l + v / 2, i, f"{v:.0f}%", ha="center", va="center", fontsize=10,
                        color="white" if name == "Poor" else INK, weight="bold")
        left += vals.values
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("Share of results (%)", color=MUTED)
    ax.set_title(title, loc="left", fontsize=13, weight="bold", color=INK)
    ax.legend(ncol=3, loc="upper left", bbox_to_anchor=(0, -0.18 if len(t) > 2 else -0.3),
              frameon=False, fontsize=9)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(left=False)
    if note:
        fig.text(0.01, 0.005, note, fontsize=8, color=MUTED)
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/{fname}", dpi=200, bbox_inches="tight")
    plt.close(fig)


NOTE = ("Relevance = average of 3+ human ratings (1 = not, 2 = mildly, 3 = highly relevant). "
        "Poor = average below 2. Perfect = average of 3 (every rater: highly relevant).")
# Chart 06 - codes customers type. Two kinds, kept separate on purpose:
#   * Spec codes (PAR38, MR16, M18, R13, T12): describe a TYPE of product and ARE usually in titles
#   * Model numbers (LW5200E, HB4190): identify ONE product and are almost never in the catalog
spec = train[train["q_norm"].map(lambda q: bool(specs_in(q))) & ~train["is_code"]].copy()
spec["code_group"] = np.where(
    [any(s in compact(t) for s in specs_in(q)) for q, t in zip(spec["q_norm"], spec["t_norm"])],
    "Spec code in title (e.g. PAR38, M18)", "Spec code not in title")
model_missing = code[code["code_match"].str.startswith("4")].assign(code_group="Model number not on product")
code_compare = outcome_table(pd.concat([spec, model_missing]), "code_group")
code_compare = code_compare.loc[["Spec code in title (e.g. PAR38, M18)", "Spec code not in title",
                                 "Model number not on product"]]
print("\nCodes customers type - matched in title vs missing:\n", code_compare)
perfect_ratio = code_compare["perfect_pct"].iloc[0] / max(code_compare["perfect_pct"].iloc[1], 0.1)
outcome_bars(code_compare,
             f"When the code a customer types is in the title, results are {perfect_ratio:.0f}x more likely to be perfect",
             "06_code_match_outcomes.png",
             f"Spec code = product-type designation (bulb PAR38/MR16, battery M18, insulation R13). Model number = one "
             f"specific item (LW5200E); only {model_coverage_pct:.0f}% of searched model numbers exist anywhere in the catalog.\n"
             + NOTE)
outcome_bars(brand_table, "When the model number is missing, the right brand limits the damage",
             "07_brand_safety_net.png", NOTE)
outcome_bars(size_table, "Size searches: the wrong size is rarely what the customer wanted",
             "08_size_match_outcomes.png", NOTE)

# Dumbbell: every category, match vs mismatch
p = penalty.sort_values("penalty")
fig, ax = plt.subplots(figsize=(9, 0.3 * len(p) + 1.8))
y = np.arange(len(p))
ax.hlines(y, p["mismatch"], p["match"], color="#bdbdbd", linewidth=2, zorder=1)
ax.scatter(p["mismatch"], y, color="#3d3d3d", s=40, zorder=2, label="Numbers don't match",
           edgecolor="white", linewidth=1.5)
ax.scatter(p["match"], y, color="#f96302", s=40, zorder=3, label="Numbers match",
           edgecolor="white", linewidth=1.5)
ax.set_yticks(y, p.index, fontsize=8)
ax.set_xlabel("Average relevance (1-3)", color=MUTED)
ax.set_title(f"Mismatched numbers score worse in {(p['penalty'] > 0).sum()} of {len(p)} categories",
             loc="left", fontsize=13, weight="bold", color=INK)
ax.legend(loc="lower right", frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#eeeeee")
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/09_category_number_penalty.png", dpi=200)
plt.close(fig)

with pd.ExcelWriter(f"{OUT_DIR}/summary_tables.xlsx", mode="a", engine="openpyxl",
                    if_sheet_exists="replace") as xw:
    code_match_table.to_excel(xw, sheet_name="code_match_levels")
    brand_table.to_excel(xw, sheet_name="brand_safety_net")
    size_table.to_excel(xw, sheet_name="size_match")
    penalty.to_excel(xw, sheet_name="category_number_penalty")
    seg.to_excel(xw, sheet_name="worst_segments")
    per_search.to_excel(xw, sheet_name="per_code_search")

In [ ]:
# %% [14] Why do poor results happen? Root-cause breakdown of every result scored below 2.0
# Each poor result gets ONE primary reason, checked in this order (first match wins), so the
# shares add up to 100%. Order = most specific / most fixable first.
#   1. Model number not on product - searched a model #, it is nowhere on the product
#   2. Wrong size / number         - a number in the search is not in the title or specs
#   3. Wrong brand                 - searched a brand, product is a different brand
#   4. Misspelled search           - search contains a likely typo
#   5. Product word missing        - the main product word of the search (e.g. "pruners") is not
#                                    in the title. Split with the Qwen match score into
#                                    "different product" (low match) vs "same thing, other words".
#   6. Related item, other reason  - words line up, but the average rating is still below 2 (needs review)
train["a_norm"] = train["product_uid"].map(attr_text).fillna("").map(normalize)
STOP_WORDS = set("the a an and or for with of in on to by from x set pack piece kit "
                 "in ft lb gal volt watt amp mm cm white black".split())


def _stem(w):
    return re.sub(r"(ies|es|s)$", "", w)


def main_product_word(q):
    """Last meaningful catalog word of the search: 'cordless drill battery' -> 'battery'."""
    words = [w for w in re.findall(r"[a-z]+", q) if len(w) >= 3 and w not in STOP_WORDS and w in vocab]
    return words[-1] if words else None


def poor_reason_flags(r):
    codes = [c.replace("-", "") for c in codes_in(r.q_norm)]
    on_product = compact(r.t_norm) + compact(r.d_norm) + compact(r.a_norm)
    # numbers in the search that are not part of a model number
    q_nums = set(NUM_RE.findall(re.sub(r"[a-z]*\d+[a-z]+[a-z0-9]*|\b\d{5,}\b", " ", r.q_norm)))
    q_brands = ngrams(r.q_norm.split()) & brands
    p_brand = brand_of.get(r.product_uid, "") or ""
    word = main_product_word(r.q_norm)
    return {
        "f_model_missing": bool(codes) and not any(c in on_product for c in codes),
        "f_wrong_number": bool(q_nums) and not q_nums <= set(NUM_RE.findall(r.t_norm + " " + r.a_norm)),
        "f_wrong_brand": bool(q_brands) and not any(b in p_brand or (p_brand and p_brand in b) or b in r.t_norm
                                                    for b in q_brands),
        "f_typo": bool(r.is_typo),
        "f_word_missing": word is not None and _stem(word) not in {_stem(w) for w in re.findall(r"[a-z]+", r.t_norm)},
    }


flags = pd.DataFrame([poor_reason_flags(r) for r in train.itertuples()], index=train.index)
train = pd.concat([train.drop(columns=flags.columns, errors="ignore"), flags], axis=1)

# Qwen split for "product word missing": low semantic match = genuinely different product
low_match = train["cos_title"] < train["cos_title"].median()
REASONS = [("f_model_missing", "Model number not on product"),
           ("f_wrong_number", "Wrong size / number"),
           ("f_wrong_brand", "Wrong brand"),
           ("f_typo", "Misspelled search")]


def primary_reason(i, r):
    for col, label in REASONS:
        if r[col]:
            return label
    if r["f_word_missing"]:
        return "Different product type" if low_match[i] else "Same product, different wording"
    return "Related item, other reason"


train["poor_reason"] = [primary_reason(i, r) for i, r in train[flags.columns].iterrows()]
poor_rows = train[train["poor"]]
poor_table = (poor_rows["poor_reason"].value_counts().rename("poor_results").to_frame()
              .assign(share_of_poor_pct=lambda d: (100 * d["poor_results"] / len(poor_rows)).round(1)))
by_reason = train.groupby("poor_reason").agg(all_results=("relevance", "size"),
                                             poor_rate_pct=("poor", "mean"), perfect_rate_pct=("perfect", "mean"))
poor_table = poor_table.join(by_reason)
poor_table[["poor_rate_pct", "perfect_rate_pct"]] = (100 * poor_table[["poor_rate_pct", "perfect_rate_pct"]]).round(1)
print(f"\nPoor results (relevance < 2.0): {len(poor_rows):,} of {len(train):,} ({100 * train['poor'].mean():.1f}%)")
print(poor_table)

# How much more common is each problem in poor results than in perfect ones? (overlapping flags)
perfect_rows = train[train["perfect"]]
lift = pd.DataFrame({
    "in_poor_pct": 100 * poor_rows[flags.columns].mean(),
    "in_perfect_pct": 100 * perfect_rows[flags.columns].mean()}).round(1)
lift["times_more_common_in_poor"] = (lift["in_poor_pct"] / lift["in_perfect_pct"].clip(lower=0.1)).round(1)
print("\nProblem frequency, poor vs perfect results:\n", lift)

poor_examples = (poor_rows.sample(frac=1, random_state=0).groupby("poor_reason").head(8)
                 .sort_values("poor_reason")[["poor_reason", "search_term", "product_title", "relevance"]])

In [ ]:
# --- 14b. Quantified recommendations ------------------------------------------------
# "Avoidable" = how many poor results would disappear if the problem group performed like the
# matching good group (an upper-bound estimate, not a guarantee).
def rates(df):
    return len(df), df["poor"].mean(), df["perfect"].mean()


has_num = train["q_norm"].str.contains(r"\d")
num_ok = train[has_num & ~train["f_wrong_number"] & ~train["f_model_missing"]]
num_bad = train[has_num & (train["f_wrong_number"] | train["f_model_missing"])]
typo, clean = train[train["is_typo"]], train[~train["is_typo"]]
brand_s, generic_s = train[train["search_type"] == "Brand"], train[train["search_type"] == "Generic"]
right_b = train[train["is_brand"] & ~train["f_wrong_brand"]]
wrong_b = train[train["f_wrong_brand"]]
size_s = train[train["is_dimension"] & ~train["is_code"]]

rec_rows = []
for lever, good, bad, label_good, label_bad in [
        ("Numbers match the product", num_ok, num_bad, "numbers match", "a number is wrong/missing"),
        ("Honor brand intent", right_b, wrong_b, "right brand", "wrong brand"),
        ("Spelling correction", clean, typo, "spelled correctly", "misspelled")]:
    ng, pg, fg = rates(good)
    nb, pb, fb = rates(bad)
    rec_rows.append({"lever": lever, "group_ok": label_good, "ok_results": ng,
                     "ok_poor_pct": round(100 * pg, 1), "ok_perfect_pct": round(100 * fg, 1),
                     "group_problem": label_bad, "problem_results": nb,
                     "problem_poor_pct": round(100 * pb, 1), "problem_perfect_pct": round(100 * fb, 1),
                     "avoidable_poor": round(nb * (pb - pg)), "extra_perfect": round(nb * (fg - fb))})
rec_table = pd.DataFrame(rec_rows).set_index("lever")
print("\nQuantified levers:\n", rec_table.T)
print(f"\nSearch volume (unseen test searches): misspelled {100 * (test_types['search_type'] == 'Misspelled').mean():.1f}% | "
      f"contain a number {100 * test['q_norm'].str.contains(r'[0-9]').mean():.1f}% | "
      f"brand {100 * test_types['is_brand'].mean():.1f}%")
print(f"Size/spec searches returning a wrong size: {100 * size_s['f_wrong_number'].mean():.0f}%")
print(f"Product word missing from title: poor {100 * train.loc[train['f_word_missing'], 'poor'].mean():.1f}% vs "
      f"{100 * train.loc[~train['f_word_missing'], 'poor'].mean():.1f}% when present")

In [ ]:
# --- Chart: what is behind poor results ----------------------------------------------
pt = poor_table.sort_values("poor_results")
fig, ax = plt.subplots(figsize=(9, 0.55 * len(pt) + 1.6))
ax.barh(pt.index, pt["share_of_poor_pct"], color="#3d3d3d", height=0.62)
for i, (s, n, r) in enumerate(zip(pt["share_of_poor_pct"], pt["poor_results"], pt["poor_rate_pct"])):
    ax.text(s + 0.5, i, f"{s:.0f}%  ({n:,} results; {r:.0f}% of this group is poor)",
            va="center", fontsize=9, color=INK)
ax.set_xlim(0, pt["share_of_poor_pct"].max() * 1.9)
ax.set_xlabel("Share of all poor results (%)", color=MUTED)
ax.set_title(f"What is behind the {len(poor_rows):,} poor results", loc="left",
             fontsize=13, weight="bold", color=INK)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, 0.005, "Poor = average rating below 2 (scale 1-3). Each poor result is assigned one primary reason, "
         "checked top-down: model # > size/number > brand > spelling > product word.", fontsize=8, color=MUTED)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/10_poor_result_reasons.png", dpi=200, bbox_inches="tight")
plt.close(fig)

with pd.ExcelWriter(f"{OUT_DIR}/summary_tables.xlsx", mode="a", engine="openpyxl",
                    if_sheet_exists="replace") as xw:
    poor_table.to_excel(xw, sheet_name="poor_reasons")
    lift.to_excel(xw, sheet_name="poor_vs_perfect_flags")
    rec_table.to_excel(xw, sheet_name="quantified_levers")
    poor_examples.to_excel(xw, sheet_name="poor_examples", index=False)

In [ ]:
# %% [15] Leadership view: three themes, quantified
# Every table below answers one business question in plain terms:
#   Results    = rated search results (search term + product shown)
#   Avg rating = average human rating, 1 (not relevant) to 3 (highly relevant)
#   Poor %     = share of results with an average rating below 2
#   Perfect %  = share of results every rater scored "highly relevant" (average of 3)
try:
    from IPython.display import display
except ImportError:          # plain `python script.py`
    display = print


def kpi_row(df, **extra):
    row = {"Results": len(df), "Avg rating (1-3)": round(df["relevance"].mean(), 2),
           "Poor %": round(100 * df["poor"].mean(), 1), "Perfect %": round(100 * df["perfect"].mean(), 1)}
    row.update(extra)
    return row


def kpi_table(groups):
    """groups: {label: dataframe or (dataframe, {extra columns})}"""
    rows = {}
    for label, g in groups.items():
        df, extra = g if isinstance(g, tuple) else (g, {})
        rows[label] = kpi_row(df, **extra)
    return pd.DataFrame(rows).T.astype({"Results": int})


def as_outcome(t):
    """Adapt a KPI table to the outcome_bars() chart helper."""
    return t.rename(columns={"Results": "rows", "Poor %": "poor_pct", "Perfect %": "perfect_pct"})


# Where does the number a customer typed show up on the product? (numbers that are not model #s)
def search_numbers(q):
    return set(NUM_RE.findall(re.sub(r"[a-z]*\d+[a-z]+[a-z0-9]*|\b\d{5,}\b", " ", q)))


def number_location(r):
    qn = search_numbers(r.q_norm)
    if not qn:
        return None
    in_title = set(NUM_RE.findall(r.t_norm))
    if qn <= in_title:
        return "In title"
    if qn <= in_title | set(NUM_RE.findall(r.a_norm)):
        return "Only in specs (not title)"
    return "Not on product"


train["number_location"] = [number_location(r) for r in train.itertuples()]
num_in_title = train[train["number_location"] == "In title"]
search_share = lambda label: round(100 * (test_types["search_type"] == label).mean(), 1)

In [ ]:
# --- Scorecard ------------------------------------------------------------------------
scorecard = pd.DataFrame({"Value": {
    "Search results rated": f"{len(train):,}",
    "Distinct customer searches": f"{train['search_term'].nunique():,}",
    "Average rating (1-3)": f"{train['relevance'].mean():.2f}",
    "Poor results (avg below 2)": f"{100 * train['poor'].mean():.1f}%  (about 1 in {round(1 / train['poor'].mean())})",
    "Perfect results (avg of 3)": f"{100 * train['perfect'].mean():.1f}%  (about 1 in {round(1 / train['perfect'].mean())})",
    "Searches containing a number": f"{100 * test['q_norm'].str.contains(r'[0-9]').mean():.0f}% of new searches",
    "Searches with a likely typo": f"{100 * test_types['is_typo'].mean():.0f}% of new searches",
    "Searches naming a brand": f"{100 * test_types['is_brand'].mean():.0f}% of new searches",
    "Model numbers searched that exist in the catalog": f"{model_coverage_pct:.0f}% ({len(in_catalog)} of {len(searched_models)})",
}})
print("\n=== SCORECARD: how search performs today ===")
display(scorecard)

In [ ]:
# --- Theme 1: Product data quality ------------------------------------------------------
theme1 = kpi_table({
    "Number in search appears in the title": num_in_title,
    "Number only in specs/attributes, not title": train[train["number_location"] == "Only in specs (not title)"],
    "Number not on the product at all": train[train["number_location"] == "Not on product"],
    "Spec code in title (PAR38, M18, R13...)": spec[spec["code_group"].str.startswith("Spec code in title")],
    "Spec code not in title": spec[spec["code_group"] == "Spec code not in title"],
    "Model number searched, not on product": model_missing,
})
print("\n=== THEME 1 - Product data quality: numbers customers type must be on the product ===")
display(theme1)

In [ ]:
# --- Theme 2: Search intent ----------------------------------------------------------------
type_order = ["Brand", "Generic", "Size/spec", "Misspelled", "Model/product code"]
theme2 = kpi_table({t: (train[train["search_type"] == t], {"Share of new searches %": search_share(t)})
                    for t in type_order})
theme2_brand = kpi_table({
    "Brand searched - same brand returned": right_b,
    "Brand searched - different brand returned": wrong_b,
})
print("\n=== THEME 2 - Search intent: results by type of search ===")
display(theme2)
display(theme2_brand)

In [ ]:
# --- Theme 3: Search assistance --------------------------------------------------------------
theme3 = kpi_table({
    "Spelling: spelled correctly": clean,
    "Spelling: misspelled": (typo, {"Share of new searches %": round(100 * test_types["is_typo"].mean(), 1)}),
    "Size/spec search: right size returned": size_s[~size_s["f_wrong_number"]],
    "Size/spec search: wrong size returned": size_s[size_s["f_wrong_number"]],
    "Product word (e.g. 'pruners') in title": train[~train["f_word_missing"]],
    "Product word missing from title": train[train["f_word_missing"]],
})
theme3["Share of new searches %"] = theme3["Share of new searches %"].astype(object).where(
    theme3["Share of new searches %"].notna(), "")
print("\n=== THEME 3 - Search assistance: spelling, measurements, product wording ===")
display(theme3)

In [ ]:
# --- Opportunity sizing ------------------------------------------------------------------------
# Each result is assigned to ONE issue (first match in this order), so the numbers add up
# without double counting. "Benchmark" = how comparable results perform when that issue is absent.
# Estimates are upper bounds: they assume fixed results would perform like the benchmark group.
LEVERS = [
    ("Model number not in catalog", lambda d: d["f_model_missing"], num_in_title,
     "Add manufacturer model numbers to the catalog as a searchable field", "Catalog / merchandising data"),
    ("Number only in specs, not title", lambda d: d["number_location"] == "Only in specs (not title)", num_in_title,
     "Index spec attributes in search; show key size/spec in the title", "Search engineering + catalog"),
    ("Wrong size / number returned", lambda d: d["number_location"] == "Not on product", num_in_title,
     "Rank exact number matches first; size filters and 'Did you mean 36 in.?' prompts", "Search ranking + UX"),
    ("Wrong brand returned", lambda d: d["f_wrong_brand"], right_b,
     "When a brand is typed, boost or require that brand", "Search ranking"),
    ("Misspelled search", lambda d: d["is_typo"], clean,
     "Spelling correction and 'Did you mean...?' suggestions", "Search UX"),
    ("Product word missing from title", lambda d: d["f_word_missing"], train[~train["f_word_missing"]],
     "Synonym lists and product-type matching (searched 'pruners', got a chainsaw). "
     "Estimate - includes some synonym gaps; confirm with a sample review", "Search relevance"),
]
remaining = pd.Series(True, index=train.index)
opp_rows = []
for name, mask_fn, bench, action, owner in LEVERS:
    mask = mask_fn(train).fillna(False).astype(bool) & remaining
    remaining &= ~mask
    grp = train[mask]
    bp, bf = bench["poor"].mean(), bench["perfect"].mean()
    opp_rows.append({
        "Issue": name, "Results affected": len(grp),
        "Share of all results %": round(100 * len(grp) / len(train), 1),
        "Poor % today": round(100 * grp["poor"].mean(), 1), "Poor % benchmark": round(100 * bp, 1),
        "Perfect % today": round(100 * grp["perfect"].mean(), 1), "Perfect % benchmark": round(100 * bf, 1),
        "Poor results avoidable": int(max(0, round(len(grp) * (grp["poor"].mean() - bp)))),
        "Extra perfect results": int(max(0, round(len(grp) * (bf - grp["perfect"].mean())))),
        "Recommendation": action, "Owner": owner})
opportunity = pd.DataFrame(opp_rows).set_index("Issue")
opportunity = opportunity.sort_values("Extra perfect results", ascending=False)
ranked = opportunity["Extra perfect results"].rank(ascending=False)
opportunity.insert(0, "Priority", np.select([ranked <= 2, ranked <= 4], ["High", "Medium"], "Low"))
total = {"Priority": "", "Results affected": opportunity["Results affected"].sum(),
         "Share of all results %": round(opportunity["Share of all results %"].sum(), 1),
         "Poor results avoidable": opportunity["Poor results avoidable"].sum(),
         "Extra perfect results": opportunity["Extra perfect results"].sum(),
         "Recommendation": "All levers combined (no double counting)", "Owner": ""}
opportunity.loc["TOTAL"] = pd.Series(total)
base_poor, base_perfect = train["poor"].sum(), train["perfect"].sum()
print("\n=== OPPORTUNITY SIZING (upper-bound estimates) ===")
display(opportunity)
print(f"Combined: poor results {base_poor:,} -> ~{base_poor - total['Poor results avoidable']:,} "
      f"({100 * total['Poor results avoidable'] / base_poor:.0f}% fewer); perfect results "
      f"{base_perfect:,} -> ~{base_perfect + total['Extra perfect results']:,} "
      f"(+{100 * total['Extra perfect results'] / base_perfect:.0f}%)")

In [ ]:
# --- Leadership charts ---------------------------------------------------------------------------
EXEC_NOTE = ("Avg rating = average of 3+ human ratings (1 = not, 2 = mildly, 3 = highly relevant). "
             "Poor = average below 2. Perfect = average of 3.")

t1 = theme1.iloc[:3].rename(index={"Number in search appears in the title": "In the title",
                                   "Number only in specs/attributes, not title": "Only in specs, not title",
                                   "Number not on the product at all": "Not on the product"})
outcome_bars(as_outcome(t1),
             f"Data quality: the number must be in the title - "
             f"{t1['Perfect %'].iloc[0] / max(t1['Perfect %'].iloc[1], 0.1):.0f}x more perfect results than when it sits only in specs",
             "11_theme1_data_quality.png",
             "Searches containing a size or number (e.g. '36 in. vanity', '60 amp breaker'). Where does that number "
             "appear on the product shown?\n" + EXEC_NOTE)

t2 = theme2.copy()
t2.index = [f"{i}  ({s:.0f}% of searches)" for i, s in zip(t2.index, t2["Share of new searches %"])]
outcome_bars(as_outcome(t2),
             f"Search intent: brand searches deliver the most perfect results ({theme2.loc['Brand', 'Perfect %']:.0f}%), "
             f"model-number searches the fewest ({theme2.loc['Model/product code', 'Perfect %']:.0f}%)",
             "12_theme2_search_intent.png",
             "Share of searches = share of new (unrated) customer searches of each type.\n" + EXEC_NOTE)

t3 = theme3.iloc[:4].rename(index={"Spelling: spelled correctly": "Spelled correctly",
                                   "Spelling: misspelled": "Misspelled",
                                   "Size/spec search: right size returned": "Size search - right size shown",
                                   "Size/spec search: wrong size returned": "Size search - wrong size shown"})
outcome_bars(as_outcome(t3),
             "Search assistance: misspellings and wrong sizes cut perfect results sharply",
             "13_theme3_search_assistance.png",
             f"{theme3.iloc[1]['Share of new searches %']:.0f}% of new searches contain a likely typo; "
             f"{100 * size_s['f_wrong_number'].mean():.0f}% of size searches are shown a product of a different size.\n"
             + EXEC_NOTE)

# Opportunity chart: extra perfect results and avoidable poor results per issue
opp = opportunity.drop(index="TOTAL").sort_values("Extra perfect results")
fig, ax = plt.subplots(figsize=(10, 0.8 * len(opp) + 1.8))
y = np.arange(len(opp))
ax.barh(y + 0.19, opp["Extra perfect results"], height=0.36, color="#f96302", label="More perfect results")
ax.barh(y - 0.19, opp["Poor results avoidable"], height=0.36, color="#3d3d3d", label="Fewer poor results")
for i, (p, q) in enumerate(zip(opp["Extra perfect results"], opp["Poor results avoidable"])):
    ax.text(p, i + 0.19, f"  +{p:,}", va="center", fontsize=9)
    ax.text(q, i - 0.19, f"  -{q:,}", va="center", fontsize=9)
ax.set_yticks(y, [f"{i}\n({p} priority)" for i, p in zip(opp.index, opp["Priority"])], fontsize=9)
ax.set_xlim(0, opp[["Extra perfect results", "Poor results avoidable"]].to_numpy().max() * 1.2)
ax.set_xlabel("Number of search results (estimated, upper bound)", color=MUTED)
ax.set_title(f"Where fixes pay off: up to +{total['Extra perfect results']:,} perfect and "
             f"-{total['Poor results avoidable']:,} poor results", loc="left", fontsize=13, weight="bold", color=INK)
ax.legend(loc="lower right", frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, 0.005, "Each result counted under one issue only. Estimate = results affected x (benchmark rate - "
         "current rate), where benchmark = similar results without that issue.", fontsize=8, color=MUTED)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/14_opportunity_sizing.png", dpi=200, bbox_inches="tight")
plt.close(fig)

In [ ]:
# --- Formatted Excel workbook for leadership ------------------------------------------------------
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

HEADER_FILL = PatternFill("solid", fgColor="F96302")


def write_exec_sheet(xw, df, sheet, title, note=None):
    df.to_excel(xw, sheet_name=sheet, startrow=2)
    ws = xw.sheets[sheet]
    ws["A1"] = title
    ws["A1"].font = Font(bold=True, size=13)
    if note:
        ws["A2"] = note
        ws["A2"].font = Font(italic=True, size=9, color="6B6B6B")
    for cell in ws[3]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = HEADER_FILL
        cell.alignment = Alignment(wrap_text=True, vertical="center")
    for col in range(1, ws.max_column + 1):
        width = max(len(str(ws.cell(row=r, column=col).value or "")) for r in range(3, ws.max_row + 1))
        ws.column_dimensions[get_column_letter(col)].width = min(max(width + 2, 10), 70)
    ws.freeze_panes = "B4"


DEF = "Avg rating 1-3. Poor = average rating below 2. Perfect = average of 3 (every rater: highly relevant)."
with pd.ExcelWriter(f"{OUT_DIR}/executive_tables.xlsx", engine="openpyxl") as xw:
    write_exec_sheet(xw, scorecard, "1 Scorecard", "How Home Depot search performs today")
    write_exec_sheet(xw, opportunity, "2 Opportunity sizing", "Where search fixes pay off (upper-bound estimates)",
                     "Each result is counted under one issue only, so rows add up. " + DEF)
    write_exec_sheet(xw, theme1, "3 Data quality", "Product data quality: numbers customers type must be on the product", DEF)
    write_exec_sheet(xw, theme2, "4 Search intent", "Search intent: results by type of search", DEF)
    write_exec_sheet(xw, theme2_brand, "4b Brand match", "When a customer names a brand", DEF)
    write_exec_sheet(xw, theme3, "5 Search assistance", "Search assistance: spelling, sizes, product wording", DEF)
    write_exec_sheet(xw, poor_table, "6 Why results are poor", "Primary reason behind each poor result", DEF)

print(f"\nDone. Charts and tables saved to {OUT_DIR}")